In [0]:
# Loading Dataset 1
df1 = spark.read.csv("/Volumes/workspace/default/aqi_data/aqi.csv", header=True, inferSchema=True)

# Loading Dataset 2
city_day = spark.read.csv("/Volumes/workspace/default/aqi_data/city_day.csv", header=True, inferSchema=True)
city_hour = spark.read.csv("/Volumes/workspace/default/aqi_data/city_hour.csv", header=True, inferSchema=True)
station_day = spark.read.csv("/Volumes/workspace/default/aqi_data/station_day.csv", header=True, inferSchema=True)
station_hour = spark.read.csv("/Volumes/workspace/default/aqi_data/station_hour.csv", header=True, inferSchema=True)
stations = spark.read.csv("/Volumes/workspace/default/aqi_data/stations.csv", header=True, inferSchema=True)

# Preview
df1.show(5)
city_day.show(5)

+----------+--------------+--------+-----------------------------+--------------------+---------+------------------+--------------------+----+
|      date|         state|    area|number_of_monitoring_stations|prominent_pollutants|aqi_value|air_quality_status|                unit|note|
+----------+--------------+--------+-----------------------------+--------------------+---------+------------------+--------------------+----+
|2025-04-30|   Maharashtra|Amravati|                            2|                PM10|       78|      Satisfactory|number_of_monitor...|NULL|
|2025-04-30|         Bihar|  Purnia|                            1|                  CO|       56|      Satisfactory|number_of_monitor...|NULL|
|2025-04-30|Madhya Pradesh|   Katni|                            1|                  O3|       98|      Satisfactory|number_of_monitor...|NULL|
|2025-04-30|  Chhattisgarh| Tumidih|                            1|                PM10|      103|          Moderate|number_of_monitor...|NULL|

In [0]:
# Dropping nulls
df1 = df1.dropna()
city_day = city_day.dropna()
station_day = station_day.dropna()

# Rename problematic column
city_day = city_day.withColumnRenamed("PM2.5", "PM25")
station_day = station_day.withColumnRenamed("PM2.5", "PM25")

# Convert Datetime column to timestamp format 
from pyspark.sql.functions import col
city_day = city_day.withColumn("Datetime", col("Datetime").cast("timestamp"))
station_day = station_day.withColumn("Datetime", col("Datetime").cast("timestamp"))

# Check schema
city_day.printSchema()

root
 |-- City: string (nullable = true)
 |-- Datetime: timestamp (nullable = true)
 |-- PM25: double (nullable = true)
 |-- PM10: double (nullable = true)
 |-- NO: double (nullable = true)
 |-- NO2: double (nullable = true)
 |-- NOx: double (nullable = true)
 |-- NH3: double (nullable = true)
 |-- CO: double (nullable = true)
 |-- SO2: double (nullable = true)
 |-- O3: double (nullable = true)
 |-- Benzene: double (nullable = true)
 |-- Toluene: double (nullable = true)
 |-- Xylene: double (nullable = true)
 |-- AQI: double (nullable = true)
 |-- AQI_Bucket: string (nullable = true)



In [0]:
from pyspark.sql.functions import col

# Creating a custom pollution index and filtering records with high AQI for analysis
# Creating pollution index
city_day = city_day.withColumn(
    "pollution_index",
    col("PM25") + col("NO2") + col("SO2") + col("CO")
)

# Filter high pollution
high_pollution = city_day.filter(col("AQI") > 200)

high_pollution.show(5)

+---------+-------------------+-----+-----+-----+-----+-----+----+----+----+-----+-------+-------+------+-----+------------+------------------+
|     City|           Datetime| PM25| PM10|   NO|  NO2|  NOx| NH3|  CO| SO2|   O3|Benzene|Toluene|Xylene|  AQI|  AQI_Bucket|   pollution_index|
+---------+-------------------+-----+-----+-----+-----+-----+----+----+----+-----+-------+-------+------+-----+------------+------------------+
|    Delhi|2015-01-01 00:00:00|153.3|241.7|182.9| 33.0| 81.3|38.5|1.87|64.5| 83.6|  18.93|  20.81|  8.32|204.5|      Severe|252.67000000000002|
|  Chennai|2015-01-01 00:00:00|174.1|275.4| 56.2| 68.8|230.9|28.5|8.56|60.8| 43.9|  19.07|  10.19|  9.63|486.5|      Severe|            312.26|
|Bangalore|2015-01-01 00:00:00|171.6|117.7|123.3| 12.4| 61.9|49.7|1.26|79.7|164.3|   6.04|  12.74|  9.59|489.7|        Good|            264.96|
|    Delhi|2015-01-02 00:00:00|344.2|173.2|179.8|128.4|156.1|15.6|6.28| 7.5|  9.4|   0.55|   15.6|  2.45|238.5|Satisfactory|            

In [0]:
# Join station_day with stations metadata
joined_df = station_day.join(stations, on=["City", "Station"])

joined_df.show(5)

+-------+----------+-------------------+-----+-----+-----+----+-----+----+----+----+-----+-------+-------+------+-----+----------+
|   City|   Station|           Datetime| PM25| PM10|   NO| NO2|  NOx| NH3|  CO| SO2|   O3|Benzene|Toluene|Xylene|  AQI|AQI_Bucket|
+-------+----------+-------------------+-----+-----+-----+----+-----+----+----+----+-----+-------+-------+------+-----+----------+
|  Delhi|Station_D1|2015-01-01 00:00:00| 58.3|223.2|126.6|85.5|207.5|18.4|0.25|42.7|184.9|   8.03|  12.72|   4.1|266.4|  Moderate|
|  Delhi|Station_D2|2015-01-01 00:00:00|222.5|541.4|198.6|20.8|  0.6|25.5|6.21|95.0| 76.2|  17.69|  13.85|  2.79|185.3|      Poor|
| Mumbai|Station_M1|2015-01-01 00:00:00| 36.6|160.8|164.2|18.6| 94.8| 8.9|6.84|70.7|195.4|   5.53|  26.93|  2.64|  5.9|  Moderate|
| Mumbai|Station_M2|2015-01-01 00:00:00|368.8|526.2|140.6|74.4|153.2|15.2|4.03|61.0|164.2|   1.62|   9.35|  3.27|176.7|      Poor|
|Chennai|Station_C1|2015-01-01 00:00:00|188.6| 88.8| 87.5|58.0|  6.1|48.3|9.66|46.2

In [0]:
city_day.groupBy("City").avg("AQI").orderBy("avg(AQI)", ascending=False).show()

+---------+------------------+
|     City|          avg(AQI)|
+---------+------------------+
|   Mumbai|253.31905283328766|
|    Delhi| 251.4963044073366|
|  Kolkata|250.63539556528883|
|  Chennai|250.33876266082672|
|Bangalore|  249.767396660279|
+---------+------------------+



In [0]:
city_day.groupBy("City").max("AQI").orderBy("max(AQI)", ascending=False).show()

+---------+--------+
|     City|max(AQI)|
+---------+--------+
|   Mumbai|   500.0|
|    Delhi|   499.9|
|  Kolkata|   499.9|
|Bangalore|   499.9|
|  Chennai|   499.8|
+---------+--------+



In [0]:
city_day.groupBy("AQI_Bucket").count().show()

+------------+-----+
|  AQI_Bucket|count|
+------------+-----+
|Satisfactory| 3098|
|        Good| 3094|
|    Moderate| 2947|
|      Severe| 3121|
|   Very Poor| 2989|
|        Poor| 3016|
+------------+-----+



In [0]:
from pyspark.sql.functions import year

city_day.withColumn("year", year("Datetime")) \
    .groupBy("year") \
    .avg("AQI") \
    .orderBy("year") \
    .show()

+----+------------------+
|year|          avg(AQI)|
+----+------------------+
|2015|247.41802739726054|
|2016|250.10579234972653|
|2017|250.88679452054797|
|2018| 246.3123287671231|
|2019|255.42964383561608|
|2020|247.30775956284123|
|2021|252.50273972602702|
|2022|250.47430136986327|
|2023|257.00613698630184|
|2024| 253.6764480874317|
+----+------------------+



In [0]:
display(city_day.groupBy("City").avg("AQI"))

City,avg(AQI)
Delhi,251.4963044073366
Chennai,250.33876266082672
Kolkata,250.63539556528883
Bangalore,249.767396660279
Mumbai,253.31905283328766


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
city_day.createOrReplaceTempView("aqi_data")

spark.sql("""
SELECT City, AVG(AQI) as avg_aqi
FROM aqi_data
GROUP BY City
ORDER BY avg_aqi DESC
""").show()

+---------+------------------+
|     City|           avg_aqi|
+---------+------------------+
|   Mumbai|253.31905283328766|
|    Delhi| 251.4963044073366|
|  Kolkata|250.63539556528883|
|  Chennai|250.33876266082672|
|Bangalore|  249.767396660279|
+---------+------------------+



In [0]:
# Big Data AQI Analysis Project

# This project demonstrates a complete Big Data pipeline using Apache Spark on Databricks.

# Steps performed:
# - Data ingestion from multiple AQI datasets
# - Data preprocessing including handling missing values and data type conversion
# - Data transformation through feature engineering and filtering
# - Data integration using joins across datasets
# - Data analysis using aggregation, grouping, and SQL queries

# Apache Spark enables distributed processing of large-scale air quality data efficiently, while Databricks provides a unified platform for data engineering and analysis.